### 構造生成

ランダム変異を加えて構造を生成する。


In [ ]:
import sys
import os
import numpy as np

from pymatgen.core import Lattice, Structure
from pymatgen.io.cif import CifWriter, CifParser
from pymatgen.io.xcrysden import XSF


In [ ]:
np.random.seed(0)

In [ ]:
g_path_prefix = "../data/Fe_structure"
g_files = [
    'Fe-bcc.cif',
    'Fe-fcc.cif',
    'Fe-hcp.cif'
]

In [ ]:
def modify_cell(structure, noise_abc=0.02, noise_angle=0.03, noise_cart=0.05):
    """add small displacement, distortion to the structure

    Args:
        structure (Structure): crystal structure
        noise_abc (float, optional): noise to abc length. Defaults to 0.02.
        noise_angle (float, optional): noise to angles. Defaults to 0.02.
        nise_cart (float, optional): noise to cartesian coordinates. Defaults to 0.1.
        
    Returns:
        Structure: noised Structure
    """
    noise_abc_ = np.random.normal(loc=0.0, scale=noise_abc, size=3)   # N(0.0, noise_abc^2)
    noise_angle_ = np.random.normal(loc=0.0, scale=noise_angle, size=3) # N(0.0, noise_angle^2)
    noise_cart_ = np.random.random(size=1)*noise_cart  # uniform random value of [0, noise_cart]
    print("noise, abc, angle, cart", noise_abc_, noise_angle_, noise_cart_)
    
    abc = structure.lattice.abc
    abc += abc*noise_abc_

    angles = structure.lattice.angles
    angles += angles*noise_angle_

    print("abc", abc, "angles", angles)

    # lattice = Lattice.from_lengths_and_angles(abc, angles)
    lattice = Lattice.from_parameters(a=abc[0], b=abc[1], c=abc[2],
                                      alpha=angles[0], beta=angles[1], gamma=angles[2])
    
    distorted_structure = Structure(lattice, structure.species, structure.frac_coords)
    
    # add noise to cart_coords
    r = (np.random.random(size=distorted_structure.cart_coords.shape)-0.5)*noise_cart_
    cart_coords = distorted_structure.cart_coords + r
    
    struct = Structure(lattice, distorted_structure.species, cart_coords, coords_are_cartesian=True)
    
    return struct

copy original structures

In [ ]:
import shutil
g_random_dir = "Fe_random_calculated"
os.makedirs(g_random_dir, exist_ok=True)

for _name in g_files:
    _inputfile = os.path.join(g_path_prefix, _name)
    _outputfile = os.path.join(g_random_dir, _name)
    shutil.copyfile(_inputfile, _outputfile)

In [ ]:
def make_distorted_supercell(files, random_dir, nstructure: int = 20):
    """構造ファイルを読み、supercellをつくり、distortion, displacementを加える。

    Args:
        files ([str]): filenames
        random_dir (str): directory name to place the structures
        nstructure (int, optional): the number of structures. Defaults to 20.
    """
    for name in files:
        print()
        print(name)
        print()
        s = name.split(".")
        key = s[0]
        prototype = s[0].split("-")[0]
        if prototype=="hcp":
            supercell=[2,2,1]
        else:
            supercell=[2,2,2]
        cifparser = CifParser(os.path.join(random_dir,name))
        structure = cifparser.get_structures(primitive=False)[0]
        structure.make_supercell(supercell)
        for i in range(nstructure):
            struc = modify_cell(structure)
            name2 = "{}/{}_{:02}.cif".format(random_dir, key, i)
            print(name2)
            struc.to(fmt="cif", filename=name2)
            i += 1
            
g_nstructure = 15
make_distorted_supercell(g_files, g_random_dir, nstructure=g_nstructure)

In [ ]:
import json
with open("condition.json","w") as f:
    json.dump({"nstructure":g_nstructure},f)


In [ ]:
"done"